# 1. CombineClassifier

In [1]:
import time
import numpy as np
from sklearn.datasets import make_classification
from sklearn.metrics import accuracy_score
from cobra.combine_classifier import CombineClassifier

X, y = make_classification(n_samples=10000, n_features=20, n_classes=2, random_state=42)
X_train, X_test = X[:8000], X[8000:]
y_train, y_test = y[:8000], y[8000:]

In [2]:
import numpy as np

def generate_estimators(n_total=30):
    estimators = []

    # Logistic Regression (8)
    for c in np.logspace(-2, 2, 8):
        estimators.append(("logistic_regression", {"C": c}))

    # Random Forest (8)
    for n in [10, 50, 100, 150, 200, 250, 300, 400]:
        estimators.append(("random_forest_classifier", {"n_estimators": n}))

    # SVC (7)
    for g in np.logspace(-3, 1, 7):
        estimators.append(("svc", {"gamma": g}))

    # KNN (7)
    for k in [3, 5, 7, 9, 11, 13, 15]:
        estimators.append(("k_neighbors_classifier", {"n_neighbors": k}))

    return estimators[:n_total]

In [3]:
# old
model1 = CombineClassifier()
start = time.time()
model1.fit(X_train, y_train)
fit_time = time.time() - start

start = time.time()
y_pred1 = model1.predict(X_test)
pred_time = time.time() - start

print(f"Original fit: {fit_time:.3f}s, predict: {pred_time:.3f}s, accuracy : {accuracy_score(y_test, y_pred1)}")

Original fit: 0.865s, predict: 0.651s, accuracy : 0.924


In [4]:
# new
from cobra.combine_classifier_v2 import CombineClassifier as CombineClassifierV2
# Test original
model = CombineClassifierV2()
start = time.time()
model.fit(X_train, y_train)
fit_time = time.time() - start

start = time.time()
y_pred = model.predict(X_test)
pred_time = time.time() - start

print(f"Original fit: {fit_time:.3f}s, predict: {pred_time:.3f}s, accuracy : {accuracy_score(y_test, y_pred)}")

Original fit: 0.876s, predict: 0.300s, accuracy : 0.9245


In [5]:
estimators = generate_estimators(30)

In [6]:
model = CombineClassifier(
    estimators=estimators
)
start = time.time()
model.fit(X_train, y_train)
fit_time = time.time() - start

start = time.time()
y_pred = model.predict(X_test)
pred_time = time.time() - start

print(f"Original fit: {fit_time:.3f}s, predict: {pred_time:.3f}s, accuracy : {accuracy_score(y_test, y_pred)}")

Original fit: 30.365s, predict: 2.367s, accuracy : 0.9135


In [7]:
model = CombineClassifierV2(
    estimators=estimators
)
start = time.time()
model.fit(X_train, y_train)
fit_time = time.time() - start

start = time.time()
y_pred = model.predict(X_test)
pred_time = time.time() - start

print(f"Original fit: {fit_time:.3f}s, predict: {pred_time:.3f}s, accuracy : {accuracy_score(y_test, y_pred)}")

Original fit: 30.587s, predict: 2.408s, accuracy : 0.914


In [8]:
from cobra.combine_classifier_v2 import CombineClassifierFast

model = CombineClassifierFast(
    estimators=estimators,
    use_faiss=True,
    faiss_k=10
)
start = time.time()
model.fit(X_train, y_train)
fit_time = time.time() - start

start = time.time()
y_pred = model.predict(X_test)
pred_time = time.time() - start

print(f"Original fit: {fit_time:.3f}s, predict: {pred_time:.3f}s, accuracy : {accuracy_score(y_test, y_pred)}")

Original fit: 30.984s, predict: 2.035s, accuracy : 0.9175


In [9]:
from cobra.core.estimators.base import EstimatorFactory

estimators = generate_estimators(30)
models = {}

models["combine_classifier"] = CombineClassifierFast(
    estimators=estimators,
    use_faiss=True,
    faiss_k=10
)

for index, (name, params) in enumerate(estimators):
    model = EstimatorFactory.create(name, **(params or {}))
    models[f"{index + 1}{name}"] = model

results = []

for name, model in models.items():
    # ---- train ----
    start = time.time()
    model.fit(X_train, y_train)
    fit_time = time.time() - start
    # ---- predict ----
    start = time.time()
    y_pred = model.predict(X_test)
    pred_time = time.time() - start
    # ---- accuracy ----
    acc = accuracy_score(y_test, y_pred)
    results.append({
        "model": name,
        "accuracy": acc,
        "fit_time": fit_time,
        "pred_time": pred_time
    })
    print(f"{name:25s} | acc={acc:.4f} | fit={fit_time:.3f}s | pred={pred_time:.3f}s")


combine_classifier        | acc=0.9175 | fit=31.204s | pred=2.090s
1logistic_regression      | acc=0.8930 | fit=0.002s | pred=0.000s
2logistic_regression      | acc=0.8935 | fit=0.002s | pred=0.000s
3logistic_regression      | acc=0.8930 | fit=0.001s | pred=0.000s
4logistic_regression      | acc=0.8920 | fit=0.001s | pred=0.000s
5logistic_regression      | acc=0.8920 | fit=0.001s | pred=0.000s
6logistic_regression      | acc=0.8920 | fit=0.001s | pred=0.000s
7logistic_regression      | acc=0.8920 | fit=0.002s | pred=0.000s
8logistic_regression      | acc=0.8920 | fit=0.001s | pred=0.000s
9random_forest_classifier | acc=0.9330 | fit=0.140s | pred=0.002s
10random_forest_classifier | acc=0.9385 | fit=0.691s | pred=0.007s
11random_forest_classifier | acc=0.9380 | fit=1.394s | pred=0.013s
12random_forest_classifier | acc=0.9385 | fit=2.011s | pred=0.019s
13random_forest_classifier | acc=0.9380 | fit=2.737s | pred=0.025s
14random_forest_classifier | acc=0.9375 | fit=3.395s | pred=0.031s
15ra

In [10]:
print("\n================ RANKING (by accuracy) ================\n")
results = sorted(results, key=lambda x: x["accuracy"], reverse=True)
for idx, r in enumerate(results):
    print(f"Rank {idx + 1} : {r['model']:25s} → acc={r['accuracy']:.4f}")


================ RANKING (by accuracy) ================

Rank 1 : 10random_forest_classifier → acc=0.9385
Rank 2 : 12random_forest_classifier → acc=0.9385
Rank 3 : 11random_forest_classifier → acc=0.9380
Rank 4 : 13random_forest_classifier → acc=0.9380
Rank 5 : 15random_forest_classifier → acc=0.9380
Rank 6 : 14random_forest_classifier → acc=0.9375
Rank 7 : 16random_forest_classifier → acc=0.9360
Rank 8 : 9random_forest_classifier → acc=0.9330
Rank 9 : 20svc                     → acc=0.9230
Rank 10 : 19svc                     → acc=0.9225
Rank 11 : combine_classifier        → acc=0.9175
Rank 12 : 18svc                     → acc=0.9115
Rank 13 : 17svc                     → acc=0.8985
Rank 14 : 28k_neighbors_classifier  → acc=0.8980
Rank 15 : 30k_neighbors_classifier  → acc=0.8970
Rank 16 : 29k_neighbors_classifier  → acc=0.8960
Rank 17 : 2logistic_regression      → acc=0.8935
Rank 18 : 1logistic_regression      → acc=0.8930
Rank 19 : 3logistic_regression      → acc=0.8930
Rank 20 : 4lo

# 2. GradientCobra

In [1]:
import time
from sklearn.metrics import mean_squared_error
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from cobra.core.estimators import EstimatorFactory

X, y = make_regression(n_samples=10000, n_features=2, noise=1, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

In [2]:
from cobra.gradientcobra import GradientCOBRA

model = GradientCOBRA()
start = time.time()
model.fit(X_train, y_train)
fit_time = time.time() - start

start = time.time()
y_pred = model.predict(X_test)
pred_time = time.time() - start
print(f"Original fit: {fit_time:.3f}s, predict: {pred_time:.3f}s, mse: {mean_squared_error(y_test, y_pred)}")

Gradient Descent: 100%|██████████| 100/100 [00:42<00:00,  2.34it/s, score=19887.174861, grad=0.001630, lr=0.010000]


Original fit: 43.508s, predict: 0.317s, mse: 9761.048727597374


In [3]:
from gradientcobra.gradientcobra import GradientCOBRA as OldGradientCOBRA

model = OldGradientCOBRA(opt_method="grid")
start = time.time()
model.fit(X_train, y_train)
fit_time = time.time() - start
start = time.time()
y_pred = model.predict(X_test)
pred_time = time.time() - start
print(f"Original fit: {fit_time:.3f}s, predict: {pred_time:.3f}s, mse: {mean_squared_error(y_test, y_pred)}")


* Grid search progress: 100%|██████████| 300/300 [00:17<00:00, 17.42it/s]


Original fit: 18.043s, predict: 0.462s, mse: 3.7449261180660858


In [4]:
linear = EstimatorFactory.create("linear_regression")
linear.fit(X_train, y_train)
y_pred = linear.predict(X_test)
print(f"mse: {mean_squared_error(y_test, y_pred)}")

mse: 0.9957692375623143


In [5]:
import numpy as np
from cobra.gradientcobra_v2 import GradientCOBRA as GradientCOBRAV2

model = GradientCOBRAV2(opt_method='grid', aggregator="wmean", random_state=42)
start = time.time()
model.fit(X_train, y_train)
fit_time = time.time() - start
start = time.time()
y_pred = model.predict(X_test)
pred_time = time.time() - start
print(f"Original fit: {fit_time:.3f}s, predict: {pred_time:.3f}s, mse: {mean_squared_error(y_test, y_pred)}")


Grid Search: 100%|██████████| 100/100 [00:04<00:00, 24.36it/s, best=5.5811]


Original fit: 7.173s, predict: 1.563s, mse: 114.64560246732968
